In [ ]:
import pandas as pd
import re

In [ ]:
gold = pd.read_excel("goldenset(3).xlsx")   #голден сет
chunks = pd.read_csv("chunks_final.csv")   # чанки

print(gold.shape)  #смотрим их размеры
print(chunks.shape)

(54, 4)
(20630, 8)


In [ ]:
def make_company_slug(name: str) -> str:
    if pd.isna(name):  #если значение пустое (Nan)
        return ""

    name = name.lower()     #переновим в нижний регистр
    name = re.sub(r'[^a-zа-я0-9]+', ' ', name)   #убираем все кроме букв
    name = name.strip(' ')  #убираем пробелы по краям

    return name
#применяем функцию к колонке с компаниями
gold["company_slug"] = gold["company"].apply(make_company_slug)

In [ ]:
def is_candidate_relevant(question: str, chunk: str) -> bool:
    """
    Проверяет, есть ли пересечение слов между вопросом и чанком
    """
    q_words = set(re.findall(r'\w+', question.lower()))  #разбиваем вопрос и чанк на слова
    c_words = set(re.findall(r'\w+', chunk.lower()))

    # пересечение слов
    overlap = q_words & c_words

    return len(overlap) >= 2  # если хотя бы 2 слова совпали - считаем релевантным

In [ ]:
rows = []  #сюда будут попадать итоговые строки

for idx, row in gold.iterrows():

    company = row["company_slug"]  #компания
    page = row["pdf_page"]            #страница пдф
    question = row["question"]       #вопрос

#выбор чанков, которые совпадают по компании и странице

    candidates = chunks[
        (chunks["company_slug"] == company) &
        (chunks["pdf_page"] == page)
    ]

    #фильтруем кандидатов
    filtered = []

    for _, ch in candidates.iterrows():
        if is_candidate_relevant(question, ch["text"]):
            filtered.append(ch)

    # если после фильтра пусто , то берем все (fallback)
    if not filtered:
        filtered = candidates.to_dict("records")
    #формируем строки для разметки
    for ch in filtered:
        rows.append({
            "question_id": idx,   #Id вопроса
            "question": question, #вопрос
            "company": row["company"],  #название компании
            "pdf_page": page,   #сраница
            "chunk_id": ch["chunk_id"],  #id чанка
            "chunk_text": ch["text"],   #текст чанка
            "is_relevant": None   #позже вручную тут будет 1 - если релевантно
        })                                               # 0 - если нет

markup = pd.DataFrame(rows)

print(markup.shape)
markup.head()

NameError: name 'gold' is not defined

In [ ]:
markup.to_excel("gold_markup3.xlsx", index=False)   #сохраняем в эксель